学习基础知识
tensors(张量)
1.直接来自数据
2.来自numpy数组
3.来自另一个张量
张量的属性
tensor.shape张量的形状，tensor.dtype里面的数字是什么类型的 tensor.device数据的位置（cpu or gpu）
默认情况下，张量是在CPU上创建的

In [ ]:
torch.stack这里还是有一点朦朦的
那个dim=0,1-1到底怎么加？有什么含义啊

dataset and dataloader
torch.utils.data.DataLoader和torch.utils.data.Dataset

transform
transform用于修改特征
target_transform修改标签

构建神经网络
获取训练设备——>定义类(继承nn.Module)——>定义模型层数——>nn.压平(nn.Flatten)——>nn.线性(nn.Linear)

自动微分torch.autograd
反向传播算法：根据损失函数相对于给定参数的梯度来调整参数或者是模型权重

loss.backward不再是魔法
雅可比矩阵（Jacobian）：多输入和多输出之间的求导
向量-雅可比乘积（VJP）
L对y的关系（V），y对x的关系（J），乘在一起（P）,得到L对x的关系
VJP最核心的价值：不把完整的Jacobian矩阵造出来，而是直接计算我们最终真正需要的结果，完美利用矩阵乘法的结合律
vT(J3​J2​J1​)——>((vTJ3​)J2​)J1​
loss.backward()链式法则，一层一层反着求梯度
w.grad最终得到loss对w的导数
optimizer.step()根据这些梯度真的修改w

通过示例学习pytorch
numpy


In [1]:
import numpy as np
import math

# Create random input and output data
x = np.linspace(-math.pi, math.pi, 2000)
y = np.sin(x)

# Randomly initialize weights
a = np.random.randn()
b = np.random.randn()
c = np.random.randn()
d = np.random.randn()

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    # y = a + b x + c x^2 + d x^3
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = np.square(y_pred - y).sum()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

print(f'Result: y = {a} + {b} x + {c} x^2 + {d} x^3')

99 214.67264243236548
199 145.9043972885334
299 100.14950258975476
399 69.69428509125858
499 49.41438885283364
599 35.90425734591825
699 26.89988241058068
799 20.89563749899107
899 16.88987584655092
999 14.21597639816909
1099 12.430105733508432
1199 11.236634649684245
1299 10.438560884436189
1399 9.904542866587892
1499 9.546971089119333
1599 9.307376026516629
1699 9.146714173428114
1799 9.038898728230151
1899 8.966489234304262
1999 8.917818340260789
Result: y = 0.005716308216156245 + 0.8649598969370546 x + -0.0009861582870662495 x^2 + -0.09449947788049828 x^3


In [2]:
import torch

dtype = torch.float
device = torch.device("cpu")
# device = torch.device("cuda:0") # Uncomment this to run on GPU

# Create random input and output data
x = torch.linspace(-torch.pi, torch.pi, 2000, device=device, dtype=dtype)
y = torch.sin(x)

# Randomly initialize weights
a = torch.randn((), device=device, dtype=dtype)
b = torch.randn((), device=device, dtype=dtype)
c = torch.randn((), device=device, dtype=dtype)
d = torch.randn((), device=device, dtype=dtype)

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = (y_pred - y).pow(2).sum().item()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights using gradient descent
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d


print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

99 1741.934814453125
199 1167.823486328125
299 784.3984375
399 528.17236328125
499 356.8420104980469
599 242.20468139648438
699 165.4489288330078
799 114.02029418945312
899 79.53662872314453
999 56.39689636230469
1099 40.85707473754883
1199 30.4124698638916
1299 23.386451721191406
1399 18.655834197998047
1499 15.467897415161133
1599 13.317499160766602
1699 11.865570068359375
1799 10.88428020477295
1899 10.220390319824219
1999 9.770755767822266
Result: y = -0.02096063643693924 + 0.8337072730064392 x + 0.0036160580348223448 x^2 + -0.09005406498908997 x^3


In [ ]:
import torch
import math

# We want to be able to train our model on an `accelerator <https://pytorch.org/docs/stable/torch.html#accelerators>`__
# such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

dtype = torch.float
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
torch.set_default_device(device)

# Create Tensors to hold input and outputs.
# By default, requires_grad=False, which indicates that we do not need to
# compute gradients with respect to these Tensors during the backward pass.
x = torch.linspace(-1, 1, 2000, dtype=dtype)#表示有2000个训练点
y = torch.exp(x) # A Taylor expansion would be 1 + x + (1/2) x**2 + (1/3!) x**3 + ...

# Create random Tensors for weights. For a third order polynomial, we need
# 4 weights: y = a + b x + c x^2 + d x^3
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors during the backward pass.
a = torch.randn((), dtype=dtype, requires_grad=True)
b = torch.randn((), dtype=dtype, requires_grad=True)
c = torch.randn((), dtype=dtype, requires_grad=True)
d = torch.randn((), dtype=dtype, requires_grad=True)

initial_loss = 1.
learning_rate = 1e-5
for t in range(5000):#这个5000表示把这2000给我数据反复训练5000次
    # Forward pass: compute predicted y using operations on Tensors.
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss using operations on Tensors.
    # Now loss is a Tensor of shape (1,)
    # loss.item() gets the scalar value held in the loss.
    loss = (y_pred - y).pow(2).sum()

    # Calculare initial loss, so we can report loss relative to it
    if t==0:
        initial_loss=loss.item()

    if t % 100 == 99:
        print(f'Iteration t = {t:4d}  loss(t)/loss(0) = {round(loss.item()/initial_loss, 6):10.6f}  a = {a.item():10.6f}  b = {b.item():10.6f}  c = {c.item():10.6f}  d = {d.item():10.6f}')

    # Use autograd to compute the backward pass. This call will compute the
    # gradient of loss with respect to all Tensors with requires_grad=True.
    # After this call a.grad, b.grad. c.grad and d.grad will be Tensors holding
    # the gradient of the loss with respect to a, b, c, d respectively.
    loss.backward()#开始反向求梯度

    # Manually update weights using gradient descent. Wrap in torch.no_grad()
    # because weights have requires_grad=True, but we don't need to track this
    # in autograd.
    with torch.no_grad():#下面的只是参数更新，别再记录了
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad

        # Manually zero the gradients after updating weights
        a.grad = None
        b.grad = None
        c.grad = None
        d.grad = None

print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

Using cuda device
Iteration t =   99  loss(t)/loss(0) =   0.372317  a =   1.391161  b =   0.434689  c =  -0.550926  d =   0.920395
Iteration t =  199  loss(t)/loss(0) =   0.207679  a =   1.283107  b =   0.522650  c =  -0.254522  d =   0.905214
Iteration t =  299  loss(t)/loss(0) =   0.122915  a =   1.204904  b =   0.561290  c =  -0.038789  d =   0.863411
Iteration t =  399  loss(t)/loss(0) =   0.076537  a =   1.148026  b =   0.590638  c =   0.118129  d =   0.819995
Iteration t =  499  loss(t)/loss(0) =   0.050572  a =   1.106656  b =   0.617036  c =   0.232264  d =   0.778701
Iteration t =  599  loss(t)/loss(0) =   0.035582  a =   1.076565  b =   0.641570  c =   0.315282  d =   0.739958
Iteration t =  699  loss(t)/loss(0) =   0.026555  a =   1.054677  b =   0.664500  c =   0.375666  d =   0.703692
Iteration t =  799  loss(t)/loss(0) =   0.020818  a =   1.038758  b =   0.685952  c =   0.419586  d =   0.669756
Iteration t =  899  loss(t)/loss(0) =   0.016942  a =   1.027178  b =   0.7060

In [ ]:
import torch
import math


class LegendrePolynomial3(torch.autograd.Function):
    """
    We can implement our own custom autograd Functions by subclassing
    torch.autograd.Function and implementing the forward and backward passes
    which operate on Tensors.
    """

    @staticmethod
    def forward(input):
        """
        In the forward pass we receive a Tensor containing the input and return
        a Tensor containing the output. Check out `Extending torch.autograd <https://docs.pytorch.org/docs/stable/notes/extending.html#extending-torch-autograd>`_
        for further details.
        """
        return 0.5 * (5 * input ** 3 - 3 * input)

    @staticmethod
    def setup_context(ctx, inputs, output):
        """
        Store input for use in the backward pass using ``ctx.save_for_backward``.
        Other objects can be stored directly as attributes on the ctx object,
        such as ``ctx.my_object = my_object``.
        """
        input, = inputs
        ctx.save_for_backward(input)

    @staticmethod
    def backward(ctx, grad_output):
        """
        In the backward pass we receive a Tensor containing the gradient of the loss
        with respect to the output, and we need to compute the gradient of the loss
        with respect to the input.
        """
        input, = ctx.saved_tensors
        return grad_output * 1.5 * (5 * input ** 2 - 1)


dtype = torch.float
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)

# Create Tensors to hold input and outputs.
# By default, requires_grad=False, which indicates that we do not need to
# compute gradients with respect to these Tensors during the backward pass.
x = torch.linspace(-math.pi, math.pi, 2000, device=device, dtype=dtype)
y = torch.sin(x)

# Create random Tensors for weights. For this example, we need
# 4 weights: y = a + b * P3(c + d * x), these weights need to be initialized
# not too far from the correct result to ensure convergence.
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors during the backward pass.
a = torch.full((), 0.0, device=device, dtype=dtype, requires_grad=True)
b = torch.full((), -1.0, device=device, dtype=dtype, requires_grad=True)
c = torch.full((), 0.0, device=device, dtype=dtype, requires_grad=True)
d = torch.full((), 0.3, device=device, dtype=dtype, requires_grad=True)

learning_rate = 5e-6
for t in range(2000):
    # To apply our Function, we use Function.apply method. We alias this as 'P3'.
    P3 = LegendrePolynomial3.apply

    # Forward pass: compute predicted y using operations; we compute
    # P3 using our custom autograd operation.
    y_pred = a + b * P3(c + d * x)

    # Compute and print loss
    loss = (y_pred - y).pow(2).sum()
    if t % 100 == 99:
        print(t, loss.item())

    # Use autograd to compute the backward pass.
    loss.backward()

    # Update weights using gradient descent
    with torch.no_grad():
        a -= learning_rate * a.grad
        b -= learning_rate * b.grad
        c -= learning_rate * c.grad
        d -= learning_rate * d.grad

        # Manually zero the gradients after updating weights
        a.grad = None
        b.grad = None
        c.grad = None
        d.grad = None

print(f'Result: y = {a.item()} + {b.item()} * P3({c.item()} + {d.item()} x)')